# Sleep Quality on Productivity - Multiple Linear Regression (MLR)

### Imports and Loading Data

In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from rulefit import RuleFit
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, RobustScaler, MaxAbsScaler, Normalizer, normalize
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error

sleepcycle_dataframe = pd.read_csv('..\\sleep_cycle_productivity.csv')

# Check that dataset is imported properly
sleepcycle_dataframe.head()

,Date,Person_ID,Age,Gender,Sleep Start Time,Sleep End Time,Total Sleep Hours,Sleep Quality,Exercise (mins/day),Caffeine Intake (mg),Screen Time Before Bed (mins),Work Hours (hrs/day),Productivity Score,Mood Score,Stress Level
0,2024-04-12,1860,32,Other,23.33,4.61,5.28,3,86,87,116,8.808920,8,3,6
1,2024-11-04,1769,41,Female,21.02,2.43,5.41,5,32,21,88,6.329833,10,3,7
2,2024-08-31,2528,20,Male,22.10,3.45,5.35,7,17,88,59,8.506306,10,9,10
3,2024-02-22,8041,37,Other,23.10,6.65,7.55,8,46,34,80,6.070240,8,4,2
4,2024-02-23,4843,46,Other,21.42,4.17,6.75,10,61,269,94,11.374994,8,7,9


In [4]:
X = sleepcycle_dataframe.drop(columns=['Sleep Quality', 'Person_ID', 'Date'], axis=1)
y = sleepcycle_dataframe['Sleep Quality']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the preprocessing steps for numerical and categorical features
numerical_columns = ['Total Sleep Hours', 'Sleep Start Time','Productivity Score', 'Exercise (mins/day)', 'Caffeine Intake (mg)', 'Screen Time Before Bed (mins)', 
                     'Work Hours (hrs/day)', 'Stress Level']
categorical_columns = ['Gender']

# Create a ColumnTransformer to apply different preprocessing to numerical and categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_columns),
        ('cat', OneHotEncoder(), categorical_columns)
    ]
)

# Create a pipeline that first preprocesses the data and then applies the SVR model 
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('cubist', RuleFit(rfmode='regress', lin_standardise=True))
])

# Define the hyperparameter grid for GridSearchCV
param_grid = {
    'cubist__max_rules': [10, 20, 30, 100, 200, 300],
    'cubist__tree_size': [1, 2, 3, 4, 5],
    'cubist__sample_fract': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    'cubist__max_rules': [10, 20, 30, 100, 200, 300]
}

# Create a GridSearchCV object to find the best hyperparameters
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

# Fit the model to the training data
grid_search.fit(X_train, y_train)

# Get the best model from the grid search
best_model = grid_search.best_estimator_

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the mean squared error and R^2 score
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Print the results
print("Best hyperparameters:", grid_search.best_params_)
print("Root Mean Squared Error:", rmse)
print("R^2 Score:", r2)

KeyboardInterrupt: 